# Schweinfurt 2045 synthetic-grid smoke run: pre vs optimized HEMS

Mixed-use grid from pylovo version 100, grid result 55 (PLZ 97422, KCID 2, BCID 9): 50 buildings (27 Residential, 18 Commercial, 5 Public). This notebook compares the pre-electrification reference with the 100% electrification optimized-HEMS case. Space heat uses the TEASER source because the INFDB ro_heat schema is not loaded in the current database.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy import text
from IPython.display import SVG, display
cwd = Path.cwd().resolve()
repo = next(x for x in (cwd, *cwd.parents) if (x / "GridExpand").is_dir())
gx = repo / "GridExpand"
post = gx / "5.postprocessing"
for x in (gx, post):
    if str(x) not in sys.path:
        sys.path.insert(0, str(x))
from common.database import SurroGridDatabase
from powerflow.comparison_data import load_synthetic_powerflow_cutoff_profile, powerflow_headline_summary_db
from plotting.powerflow_asset_plots import plot_powerflow_asset_cutoff_overview_static
manifest_path = gx / "run_logs" / "schweinfurt_2045_synthetic_smoke_grid55" / "run_manifest.json"
manifest = json.loads(manifest_path.read_text())
order = ["pre", "post-hems-optimized"]
cases = {x["model_case"]: x for x in manifest["cases"]}
assert manifest["status"] == "done" and list(cases) == order
selected = manifest["selected_grid"]
print("Completed:", manifest["finished_at"])
print("Manifest:", manifest_path)
optimized_h5 = Path(cases["post-hems-optimized"]["urbs_output_h5"])
plot_dir = post / "output" / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

## Grid and building inventory

Floor area is the pipeline building-footprint field, not gross floor area.

In [ ]:
ref = {"grid_result_id": int(selected["grid_result_id"]), "version_id": str(manifest["pylovo_version_id"]), "plz": int(selected["plz"]), "kcid": int(selected["kcid"]), "bcid": int(selected["bcid"])}
db = SurroGridDatabase()
db.pylovo_version_id = str(manifest["pylovo_version_id"])
buildings = db.read_buildings(ref)
net = db.read_pandapower_grid(ref)
overview = pd.DataFrame([{
    "Grid result": int(selected["grid_result_id"]), "Buildings": len(buildings),
    "Residential": int(buildings["building_use"].eq("Residential").sum()),
    "Residential share [%]": 100 * buildings["building_use"].eq("Residential").mean(),
    "Households": int(buildings["households"].fillna(0).sum()),
    "Occupants": int(buildings["occupants"].fillna(0).sum()),
    "Footprint [m2]": buildings["floor_area"].fillna(0).sum(),
    "Buses": len(net.bus), "Lines": len(net.line), "Line length [km]": net.line["length_km"].sum()}])
display(overview.style.hide(axis="index").format({"Residential share [%]": "{:.1f}", "Footprint [m2]": "{:,.0f}", "Line length [km]": "{:.2f}"}))
composition = buildings.groupby(["building_use", "building_type"], dropna=False).agg(
    buildings=("objectid", "count"), households=("households", "sum"), occupants=("occupants", "sum"),
    footprint_m2=("floor_area", "sum"), peak_load_kw=("peak_load_in_kw", "sum")).sort_values(["building_use", "buildings"], ascending=[True, False])
display(composition.style.format({"households": "{:,.0f}", "occupants": "{:,.0f}", "footprint_m2": "{:,.0f}", "peak_load_kw": "{:,.1f}"}))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
sns.countplot(data=buildings, y="building_type", hue="building_use", ax=axes[0], palette="Set2")
axes[0].set(title="Building composition", xlabel="Number of buildings", ylabel="Building type")
axes[0].legend(title="Use", frameon=False)
sns.histplot(data=buildings, x="floor_area", hue="building_use", multiple="stack", bins=14, ax=axes[1], palette="Set2")
axes[1].set(title="Building-footprint distribution", xlabel="Building footprint [m2]", ylabel="Number of buildings")
sns.despine()
fig.tight_layout()
building_plot_path = plot_dir / "schweinfurt_2045_synthetic_grid55_buildings.svg"
fig.savefig(building_plot_path, bbox_inches="tight")
plt.close(fig)
display(SVG(filename=str(building_plot_path)))

## Optimized HEMS PV potential and capacities by building

PV potential is the sum of the LoD2 roof-section upper bounds assigned to a building. Installed values are solved urbs capacities. Heat-pump and auxiliary-heater values retain the urbs process-capacity convention; heat buffer and battery columns are energy capacities.

In [ ]:
process_input = pd.read_hdf(optimized_h5, "/urbs_in/process")
process_capacity = pd.read_hdf(optimized_h5, "/urbs_out/MILP/cap_pro")
storage_capacity = pd.read_hdf(optimized_h5, "/urbs_out/MILP/cap_sto_c")
storage_power = pd.read_hdf(optimized_h5, "/urbs_out/MILP/cap_sto_p")
def process_by_site(predicate):
    names = process_capacity.index.get_level_values("pro").astype(str)
    return process_capacity[np.array([predicate(name) for name in names])].groupby(level="sit").sum()
def storage_by_site(series, name):
    names = series.index.get_level_values("sto").astype(str)
    return series[names == name].groupby(level="sit").sum()
pv_input = process_input[process_input["Process"].astype(str).str.startswith("Rooftop PV")]
series = {
    "PV potential [kWp]": pv_input.groupby("Site")["cap-up"].sum(),
    "PV installed [kWp]": process_by_site(lambda x: x.startswith("Rooftop PV")),
    "Battery energy [kWh]": storage_by_site(storage_capacity, "battery_private"),
    "Battery power [kW]": storage_by_site(storage_power, "battery_private"),
    "Heat pump [kW]": process_by_site(lambda x: x == "heatpump_air"),
    "Auxiliary heater [kW]": process_by_site(lambda x: x == "heatpump_booster"),
    "Heat buffer [kWhth]": storage_by_site(storage_capacity, "heat_storage"),
    "EV chargers [kW]": process_by_site(lambda x: x.startswith("charging_station"))}
asset_table = buildings[["bus", "objectid", "building_use", "building_type", "households", "occupants", "floor_area"]].copy().set_index("bus")
asset_table = asset_table.rename(columns={"objectid": "Building object ID", "building_use": "Use", "building_type": "Type", "households": "Households", "occupants": "Occupants", "floor_area": "Footprint [m2]"})
for column, values in series.items():
    asset_table[column] = values.reindex(asset_table.index).fillna(0)
asset_table["PV potential used [%]"] = np.divide(100 * asset_table["PV installed [kWp]"], asset_table["PV potential [kWp]"], out=np.zeros(len(asset_table)), where=asset_table["PV potential [kWp]"].to_numpy() > 0)
capacity_columns = ["PV potential [kWp]", "PV installed [kWp]", "PV potential used [%]", "Battery energy [kWh]", "Battery power [kW]", "Heat pump [kW]", "Auxiliary heater [kW]", "Heat buffer [kWhth]", "EV chargers [kW]"]
summary_columns = [column for column in capacity_columns if column != "PV potential used [%]"]
capacity_summary = pd.DataFrame({"Buildings with capacity": (asset_table[summary_columns] > 0).sum(), "Total": asset_table[summary_columns].sum(), "Mean per building": asset_table[summary_columns].mean(), "Maximum building": asset_table[summary_columns].max()})
display(capacity_summary.style.format("{:,.2f}"))
display(asset_table.sort_values("PV potential [kWp]", ascending=False).style.format(
    {"Households": "{:.0f}", "Occupants": "{:.0f}", "Footprint [m2]": "{:,.1f}", **{x: "{:,.2f}" for x in capacity_columns}}, na_rep="-").background_gradient(
    subset=["PV potential [kWp]", "PV installed [kWp]", "Battery energy [kWh]"], cmap="YlGn"))

## Saved model artifacts

The pre case has no urbs output by design. Every post output contains the original full-year urbs input together with reduced TSAM data and the MILP result.

In [ ]:
artifact_rows = []
for model_case in order:
    case = cases[model_case]
    for kind in ("urbs_input_h5", "urbs_output_h5"):
        value = case.get(kind)
        if value:
            f = Path(value)
            artifact_rows.append({"Model case": model_case, "Artifact": kind.removesuffix("_h5"), "File": f.name, "Size [MiB]": f.stat().st_size / 2**20, "HDF keys": len(case[kind.replace("_h5", "_keys")])})
display(pd.DataFrame(artifact_rows).set_index(["Model case", "Artifact"]).style.format({"Size [MiB]": "{:.1f}"}))

## Annual demands across model cases

Annual energy is summed from each unreduced 8,760-hour input. Profile-column counts describe building, vehicle, or commodity records and are not energy values.

In [ ]:
def demand_group(x):
    x = str(x).lower()
    if x == "electricity": return "Base electricity"
    if "space_heat" in x: return "Space heat"
    if "water_heat" in x or "hot_water" in x: return "Domestic hot water"
    if x.startswith("mobility"): return "Electric mobility"
    return x
rows = []
for case in order:
    d = pd.read_hdf(cases[case]["urbs_input_h5"], "/urbs_in/demand")
    groups = np.array([demand_group(x) for x in d.columns.get_level_values(-1)])
    for group in pd.unique(groups):
        mask = groups == group
        rows.append({"model case": case, "demand": group, "annual energy [MWh]": d.loc[:, mask].to_numpy().sum() / 1000, "columns": mask.sum()})
demands = pd.DataFrame(rows)
annual_energy = demands.pivot(index="model case", columns="demand", values="annual energy [MWh]").reindex(order)
profile_columns = demands.pivot(index="model case", columns="demand", values="columns").reindex(order)
demand_summary = pd.concat({"Annual energy [MWh]": annual_energy, "Individual profile columns [count]": profile_columns}, axis=1)
demand_summary.columns.names = ["Metric", "Demand carrier"]
display(demand_summary)

## Power-flow results

Headline metrics use the compact database summaries. The static retained-asset overview compares stage pre for the reference with stage post for the electrified case. It shows only cable loading and bus voltage.

In [ ]:
labels = {"pre": "Pre", "post-hems-optimized": "HEMS optimized"}
heads = []
profiles = []
for case in order:
    stage = "pre" if case == "pre" else "post"
    filters = dict(run_name=cases[case]["powerflow_run_name"], stage=stage, ags=selected["ags"], plz=int(selected["plz"]), kcid=int(selected["kcid"]), bcid=int(selected["bcid"]))
    x = powerflow_headline_summary_db(**filters)
    x["model case"] = case
    heads.append(x)
    x = load_synthetic_powerflow_cutoff_profile(**filters)
    x["comparison_group"] = labels[case]
    profiles.append(x)
headline = pd.concat(heads, ignore_index=True)
wanted = ["model case", "n_timesteps", "n_converged_timesteps", "n_failed_timesteps",
    "trafo_loading_max_time_percent", "trafo_loading_hours_above_100", "cable_loading_max_asset_percent",
    "cable_hours_above_100_max_asset", "voltage_min_load_bus_hour_pu", "voltage_hours_below_0_90_max_asset"]
display(headline[[x for x in wanted if x in headline]].set_index("model case").reindex(order))
profile = pd.concat(profiles, ignore_index=True)
fig = plot_powerflow_asset_cutoff_overview_static(profile, group_col="comparison_group",
    asset_cutoff_percentile=1.0, center_stat="mean", show_band=False, worst_asset_per_grid=False,
    metrics=("Cables", "Voltage"),
    title="Schweinfurt synthetic grid 55: cable loading and voltage",
    save_path=plot_dir / "schweinfurt_2045_synthetic_smoke_grid55", save_formats=("svg", "pdf"))
plt.close(fig)
display(SVG(filename=str(plot_dir / "schweinfurt_2045_synthetic_smoke_grid55.svg")))

## Interpretation guardrails

- This is a one-grid integration smoke run, not a statistical validation result.
- Post cases use six 168-hour TSAM periods (1,008 modeled timesteps); annual-demand tables use unreduced inputs.
- Space heat uses the TEASER source; switch to `infdb_ro_heat` once the INFDB ro_heat schema is loaded for the physically calibrated building-level heat profiles.